In [1]:
import os

In [2]:
def rename_files(base_dir):
    """
    Standardize ASE/VASP trajectory filenames for downstream RMG database processing.

    This function iterates over compound subdirectories inside `base_dir` and renames
    ASE trajectory filenames to a canonical format that encodes the compound name
    in the filename:

        ads60_<compound>.traj
        relax_restart_<compound>.traj
        structure_<compound>.traj

    This normalizes outputs from different VASP/ASE pipelines (e.g., vasprun_* vs *.traj)
    so that later scripts can reliably identify adsorption, relaxation, and reference
    structures by filename.

    Parameters
    ----------
    base_dir : str
        Path containing one subdirectory per compound, each holding ASE trajectory files.
    """
    for compound_name in os.listdir(base_dir):
        compound_dir = os.path.join(base_dir, compound_name)
        print(f"Processing directory: {compound_dir}")

        if os.path.isdir(compound_dir):
            old_ads60_names = ['vasprun_ads60.traj', 'ads60.traj']
            old_relax_names = ['vasprun_relax_restart.traj', 'relax_restart.traj']
            old_structure_names = ['structure.traj']

            new_ads60_name = f"ads60_{compound_name}.traj"
            new_relax_name = f"relax_restart_{compound_name}.traj"
            new_structure_name = f"structure_{compound_name}.traj"

            # Rename ads60 files
            for old_name in old_ads60_names:
                old_path = os.path.join(compound_dir, old_name)
                if os.path.exists(old_path):
                    new_path = os.path.join(compound_dir, new_ads60_name)
                    os.rename(old_path, new_path)
                    print(f"Renamed {old_path} to {new_path}")

            # Rename relax_restart files
            for old_name in old_relax_names:
                old_path = os.path.join(compound_dir, old_name)
                if os.path.exists(old_path):
                    new_path = os.path.join(compound_dir, new_relax_name)
                    os.rename(old_path, new_path)
                    print(f"Renamed {old_path} to {new_path}")

            # Rename structure files
            for old_name in old_structure_names:
                old_path = os.path.join(compound_dir, old_name)
                if os.path.exists(old_path):
                    new_path = os.path.join(compound_dir, new_structure_name)
                    os.rename(old_path, new_path)
                    print(f"Renamed {old_path} to {new_path}")

In [3]:
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Data', 'Cu111'))

In [4]:
rename_files(base_dir)

Processing directory: /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2OHX
Renamed /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2OHX/ads60.traj to /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2OHX/ads60_CH2OHX.traj
Renamed /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2OHX/relax_restart.traj to /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2OHX/relax_restart_CH2OHX.traj
Processing directory: /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2X
Renamed /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2X/ads60.traj to /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2X/ads60_CH2X.traj
Renamed /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2X/relax_restart.traj to /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data/Cu111/CH2X/relax_restart_CH2X.traj
Processing directory: /projects/westgroup/asifor.t/CO2_Reduction/Dft_data/Data

In [5]:
import pandas as pd

In [6]:
def process_vibrational_frequencies(file_path):
    """
    Parse vibrational frequencies from an Excel sheet and convert them into
    a table for later processing (one molecule per row, flattened vibrational modes).
    
    The input Excel file should contain a sheet named 'Vibrational_freq' with
    the following structure:
        - Column 0: molecule name
        - Column 1: adsorption site (ignored here)
        - Columns 2+: vibrational frequencies in cm^-1
        - Real frequencies appear first
        - Imaginary frequencies appear after a blank-column separator
        
    Imaginary frequencies are replaced by 12 cm^-1.
    
    Parameters
    ----------
    file_path : str
        Path to the Excel file containing vibrational frequency data.
        
    Returns
    -------
    processed_df : pandas.DataFrame
        Table with one row per molecule and columns:
            molecule_name, vib_freq_1, vib_freq_2, ...
    """
    xls = pd.ExcelFile(file_path)
    
    df = pd.read_excel(xls, 'Vibrational_freq', skiprows=1)
    df = df.dropna(how='all')
    
    processed_data = {}
    current_molecule = None
    current_freqs = []
    
    # We ignore the vib freqs in meVs for now
    ignore_data = False
    
    # Iterate over the rows of the dataframe
    for index, row in df.iterrows():
        # check if we need to ignore data based on headers for meV section
        if ignore_data:
            break
        
        molecule_name = row.iloc[0]
        vib_freqs = row.iloc[2:].values
        
        if molecule_name == 'molecule' and row.iloc[1] == 'site' and row.iloc[2] == 'vib freq (meV)':
            ignore_data = True
            continue
        
        if molecule_name != current_molecule:
            #If we encounter a new molecule, save the previous molecule's data
            if current_molecule is not None:
                processed_data[current_molecule] = current_freqs
            # Update the current molecule and reset frequencies
            current_molecule = molecule_name
            current_freqs = []
        
        # Separate real and imaginary frequencies
        real_freqs = []
        imaginary_freqs = []
        imaginary_found = False
        
        for freq in vib_freqs:
            if pd.isna(freq) and not imaginary_found:
                imaginary_found = True
                continue
            if not imaginary_found:
                real_freqs.append(freq)
            elif not pd.isna(freq):
                imaginary_freqs.append(freq)
        
        if imaginary_found and imaginary_freqs:
            imaginary_freqs = [12] * len(imaginary_freqs)
        else:
            imaginary_freqs = []
        
        # Combine the real and imaginary frequencies back
        all_freqs = list(real_freqs) + list(imaginary_freqs)
        current_freqs.extend(all_freqs)
    
    # Save the last molecule's data
    if current_molecule is not None:
        processed_data[current_molecule] = current_freqs
    
    # Create a new DataFrame for the processed data
    max_len = max(len(freqs) for freqs in processed_data.values())
    columns = ['molecule_name'] + [f'vib_freq_{i+1}' for i in range(max_len)]
    
    rows = []
    for molecule_name, freqs in processed_data.items():
        row_data = [molecule_name] + freqs + [None] * (max_len - len(freqs))
        rows.append(row_data)
    
    processed_df = pd.DataFrame(rows, columns=columns)
    
    return processed_df

In [7]:
fpath = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'Data', 'Cu111', 'vibrational_freq_zpe_Cu111.xlsx'))

In [8]:
processed_df = process_vibrational_frequencies(fpath)

In [9]:
processed_df

,molecule_name,vib_freq_1,vib_freq_2,vib_freq_3,vib_freq_4,vib_freq_5,vib_freq_6,vib_freq_7,vib_freq_8,vib_freq_9,...,vib_freq_15,vib_freq_16,vib_freq_17,vib_freq_18,vib_freq_19,vib_freq_20,vib_freq_21,vib_freq_22,vib_freq_23,vib_freq_24
0,COOHX,3491.891706,1467.586656,1257.644191,1084.619391,670.218434,643.586867,349.464156,265.407755,200.740342,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,OCHOX,2955.310734,1521.799069,1309.049231,1307.958281,975.356117,737.444457,285.670974,282.980140,277.715241,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,COX,1811.977961,274.964033,248.799480,244.740195,128.290592,119.900693,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,XCOXCO,1459.942693,1381.855514,648.679967,584.200606,365.589420,337.177646,310.025444,227.602046,184.232288,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,CHOX,2813.061035,1506.365209,1219.406881,670.667125,439.579926,207.386418,132.360092,114.824950,28.393499,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,COHX,3599.491313,1261.226023,1092.179509,415.043406,365.160726,352.364720,148.981451,134.330365,112.416220,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,COCHOX,2674.158809,1611.240814,1548.320473,1266.033141,854.903274,795.053310,614.746568,441.785260,289.627576,...,66.182093,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,CHOHX,3636.272043,2943.167104,1348.946277,1176.042992,1041.507066,693.774180,427.332890,315.475504,231.694339,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,OCHOCHX,3098.782600,3012.792827,1434.717600,1365.799869,1257.313597,1159.237359,996.379970,850.316328,796.624548,...,200.831115,101.067220,86.247486,47.688729,NaN,NaN,NaN,NaN,NaN,NaN
9,CH2OHX,3606.752999,2990.276490,2929.892151,1365.093170,1294.242878,1124.344238,1060.444687,879.243133,441.999890,...,83.651768,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
def save_frequencies_to_txt(df, base_dir):
    """
    Write vibrational frequency of each molecule to text format.

    For each molecule in the input DataFrame, this function creates a file

        zpe_log_<molecule>.txt

    inside the corresponding molecule directory under `base_dir`. The file
    contains a sorted list of vibrational frequencies with both meV and cm^-1
    units, in a format compatible with downstream zero-point energy (ZPE) and
    thermochemistry parsing scripts used for RMG database generation.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with columns:
            - molecule_name
            - vib_freq_1, vib_freq_2, ...
        as produced by `process_vibrational_frequencies()`.

    base_dir : str
        Path containing one subdirectory per molecule, where output files
        will be written.
    """
    # Iterate through the rows of the dataframe
    for index, row in df.iterrows():
        molecule_name = row['molecule_name']

        # Find the folder corresponding to the molecule
        molecule_folder = os.path.join(base_dir, molecule_name)

        if not os.path.exists(molecule_folder):
            print(f"Error: Folder for molecule '{molecule_name}' not found.")
            continue

        # Create the txt file path
        txt_file_path = os.path.join(molecule_folder, f'zpe_log_{molecule_name}.txt')

        # Open the file for writing
        with open(txt_file_path, 'w') as file:
            # Write the header
            file.write('---------------------\n')
            file.write('  #    meV     cm^-1\n')
            file.write('---------------------\n')

            # Collect vibrational frequencies and sort them
            freqs = [freq for freq in row[1:] if not pd.isna(freq)]  # Exclude NaNs
            freqs.sort()

            # Write the sorted vibrational frequencies with 2 decimal places
            for i, freq in enumerate(freqs, start=0):
                freq_mev = freq / 8.0655429
                file.write(f'{i:3}    {freq_mev:6.2f}    {freq:6.2f}\n')

In [11]:
save_frequencies_to_txt(processed_df, base_dir)

In [12]:
def read_zpe_data(file_path):
    """
    Read zero-point energy (ZPE) data from an Excel file and return a standardized table in text format.

    The input Excel file should contain a sheet named 'ZPE' with columns:

        - molecule : molecule / adsorbate name
        - site     : adsorption site (ignored here)
        - zpe      : zero-point energy in eV

    The site column is dropped and the remaining data are returned as a two-column
    DataFrame mapping molecule name to ZPE (in eV).

    Parameters
    ----------
    file_path : str
        Path to the Excel file containing ZPE data.

    Returns
    -------
    df_zpe : pandas.DataFrame
        DataFrame with columns:
            - molecule_name
            - zpe_eV
    """
    xls = pd.ExcelFile(file_path)
    df_zpe = pd.read_excel(xls, 'ZPE', header=0)
    df_zpe = df_zpe.drop(columns=['site'])

    # Rename columns for clarity (optional)
    df_zpe.columns = ['molecule_name', 'zpe_eV']

    return df_zpe

In [13]:
zpe_df = read_zpe_data(fpath)

In [14]:
def append_zpe_to_txt(df, base_dir):
    """
    Append total zero-point energy (ZPE) values to existing vibrational frequency logs.

    For each molecule in the input DataFrame, this function locates the corresponding
    file

        zpe_log_<molecule>.txt

    inside the molecule directory under `base_dir` and appends a final line containing
    the total zero-point energy in eV.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with columns:
            - molecule_name
            - zpe_eV
        as produced by `read_zpe_data()`.

    base_dir : str
        Path containing one subdirectory per molecule with existing
        zpe_log_<molecule>.txt files.
    """
    # Iterate through the rows of the dataframe
    for index, row in df.iterrows():
        molecule_name = row['molecule_name']
        zpe = row['zpe_eV']

        # Find the folder corresponding to the molecule
        molecule_folder = os.path.join(base_dir, molecule_name)

        if not os.path.exists(molecule_folder):
            print(f"Error: Folder for molecule '{molecule_name}' not found.")
            continue

        # Create the txt file path
        txt_file_path = os.path.join(molecule_folder, f'zpe_log_{molecule_name}.txt')

        # Check if the file exists before appending
        if not os.path.isfile(txt_file_path):
            print(f"Error: File '{txt_file_path}' not found.")
            continue

        # Open the file for appending
        with open(txt_file_path, 'a') as file:
            # Write the ZPE line at the end of the file
            file.write('---------------------\n')
            file.write(f"Zero-point energy: {zpe} eV\n")

In [15]:
append_zpe_to_txt(zpe_df, base_dir)